In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.feature_selection import mutual_info_classif
import random
from random import randint
from contextlib import redirect_stdout

# Import data

In [2]:
# paths where to load files from
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
log_dir = Path('./log/logs_final/logs_convdiv')
log_dir.mkdir(exist_ok=True, parents=True)
# path where data are saved
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'

# load from csv
data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# Super mild processing

# create subject id column
data_gp = data_gp.rename(columns={'Unnamed: 0': 'Subject'})
data_en = data_en.rename(columns={'Unnamed: 0': 'Subject'})

print(data_gp.shape)
print(data_en.shape)

# combine
data_combined = pd.concat([data_gp, data_en])

print(data_combined.shape)

(497, 805)
(305, 805)
(802, 805)


In [3]:
# check stuff
data_combined.head(3)

,Subject,caseid,weight,sample_type,screener_1ongoing,screener_2impact,screener_3depression,screener_4anxiety,screener_5attention,consent,mood_yn,FNM_Q25_1,FNM_Q25_2,FNM_Q25_3,FNM_Q25_4,FNM_Q25_5,FNM_Q25_6,FNM_Q25_955,FNM_Q25_933,FNM_Q25_open1,mood_years,FNM_Q27_1,FNM_Q27_2,FNM_Q27_3,FNM_Q27_4,FNM_Q27_5,FNM_Q27_6,FNM_Q27_7,FNM_Q27_8,FNM_Q27_9,FNM_Q27_10,FNM_Q27_11,FNM_Q27_12,FNM_Q27_13,FNM_Q27_14,FNM_Q27_15,FNM_Q27_16,FNM_Q27_17,FNM_Q27_18,FNM_Q27_19,FNM_Q27_955,FNM_Q27_933,FNM_Q27_open1,mood_bothered_orig,anxiety_yn,FNM_Q30_m_1,FNM_Q30_m_2,FNM_Q30_m_3,FNM_Q30_m_4,FNM_Q30_m_5,FNM_Q30_m_6,FNM_Q30_m_7,FNM_Q30_m_8,FNM_Q30_m_955,FNM_Q30_m_933,FNM_Q30_open1,anxiety_years,FNM_Q32_1,FNM_Q32_2,FNM_Q32_3,FNM_Q32_4,FNM_Q32_5,FNM_Q32_6,FNM_Q32_7,FNM_Q32_8,FNM_Q32_9,FNM_Q32_10,FNM_Q32_955,FNM_Q32_933,FNM_Q32_open1,anxiety_bothered_orig,attention_yn,FNM_Q35_m_1,FNM_Q35_m_2,FNM_Q35_m_3,FNM_Q35_m_933,FNM_Q35_open1,attention_years,FNM_Q37_m_1,FNM_Q37_m_2,FNM_Q37_m_3,FNM_Q37_m_4,FNM_Q37_m_5,FNM_Q37_m_6,FNM_Q37_m_7,FNM_Q37_m_8,FNM_Q37_m_9,FNM_Q37_m_955,FNM_Q37_m_933,FNM_Q37_open1,attention_bothered_orig,inattention_1,inattention_2,inattention_3,inattention_4,inattention_5,inattention_6,FNM_Q1_attn,inattention_7,inattention_8,inattention_9,hyperactivity_1,hyperactivity_2,hyperactivity_3,hyperactivity_4,hyperactivity_5,impulsivity_1,impulsivity_2,impulsivity_3,impulsivity_4,sct_1,sct_2,sct_3,sct_4,sct_5,sct_6,sct_7,sct_8,sct_9,gad_1,gad_2,gad_3,gad_4,gad_5,gad_6,gad_7,phq_1,phq_2,phq_3,phq_4,phq_5,phq_6,phq_7,phq_8,hitop157,hitop81,hitop34,hitop54,hitop243,hitop182,hitop69,hitop89,hitop50,check_moderately,hitop129,hitop265,hitop124,hitop231,hitop93,hitop67,hitop245,hitop281,hitop141,hitop40,hitop204,hitop21,hitop236,hitop280,hitop84,hitop120,hitop77,hitop92,hitop258,hitop39,hitop254,hitop215,hitop95,hitop106,hitop283,hitop16,hitop20,hitop189,hitop1,hitop136,hitop246,hitop248,hitop257,hitop114,hitop117,hitop250,hitop200,hitop160,hitop23,hitop165,hitop244,hitop9,hitop142,hitop230,hitop149,hitop247,hitop99,hitop66,hitop240,hitop222,hitop90,hitop113,hitop278,hitop203,hitop159,hitop123,hitop275,hitop268,hitop225,hitop143,hitop151,hitop181,hitop211,hitop17,hitop126,hitop5,hitop261,hitop220,check_notatall,hitop15,hitop72,hitop140,hitop109,hitop197,hitop104,todayinattention_1,todayinattention_2,todayinattention_3,todayinattention_4,todayinattention_5,todayinattention_6,todayinattention_7,todayinattention_8,todayinattention_9,todayhyperactivity_1,todayhyperactivity_2,todayhyperactivity_3,todayhyperactivity_4,todayhyperactivity_5,todayimpulsivity_1,todayimpulsivity_2,todayimpulsivity_3,todayimpulsivity_4,todaysct_1,todaysct_2,todaysct_3,todaysct_4,todaysct_5,todaysct_6,todaysct_7,todaysct_8,todaysct_9,today_na1,todaygad_1,todaygad_2,todaygad_3,...,hitop23_recontact,hitop165_recontact,hitop244_recontact,hitop9_recontact,hitop142_recontact,hitop230_recontact,hitop149_recontact,hitop247_recontact,hitop99_recontact,hitop66_recontact,hitop240_recontact,hitop222_recontact,hitop90_recontact,hitop113_recontact,hitop278_recontact,hitop203_recontact,hitop159_recontact,hitop123_recontact,hitop275_recontact,hitop268_recontact,hitop225_recontact,hitop143_recontact,hitop151_recontact,hitop181_recontact,hitop211_recontact,hitop17_recontact,hitop126_recontact,hitop5_recontact,hitop261_recontact,hitop220_recontact,check_notatall_recontact,hitop15_recontact,hitop72_recontact,hitop140_recontact,hitop109_recontact,hitop197_recontact,hitop104_recontact,todayinattention_1_recontact,todayinattention_2_recontact,todayinattention_3_recontact,todayinattention_4_recontact,todayinattention_5_recontact,todayinattention_6_recontact,todayinattention_7_recontact,todayinattention_8_recontact,todayinattention_9_recontact,todayhyperactivity_1_recontact,todayhyperactivity_2_recontact,todayhyperactivity_3_recontact,todayhyperactivity_4_recontact,todayhyperactivity_5_recontact,todayimpulsivity_1_recontact,todayimpulsivity_2_recontact,todayimpulsivity_3_recontact,todayimpulsivity_4_recontact,tod

## Replace the OG hitop measures with invariant cores

In [4]:
dep_scales = [
    'anhedonic_depression_invcore',
    'appetite_gain_invcore',
    'appetite_loss',
    'cognitive_problems',
    'hyposomnia',
    'indecisiveness',
    'insomnia',
    'shame_guilt',
    ]
dep_scales = [f'hitop_{ss}' for ss in dep_scales]
anx_scales = [
        'anxious_worry_invcore',
        'panic',
        'separation_insecurity_invcore',
        'situational_phobia',
        'social_anxiety_invcore'
    ]
anx_scales = [f'hitop_{ss}' for ss in anx_scales]

#defrag data
data_gp = data_gp.copy()
data_en = data_en.copy()
data_combined = data_combined.copy()
for data in data_gp, data_en, data_combined:
    data['hitop_anhedonic_depression_invcore'] = data['hitop39'] + data['hitop77'] + data['hitop84'] + data['hitop93'] + data['hitop182'] + data['hitop230'] + data['hitop246'] 
    data['hitop_anxious_worry_invcore'] = data['hitop34'] + data['hitop89'] + data['hitop203'] + data['hitop248']
    data['hitop_appetite_gain_invcore'] = data['hitop120'] + data['hitop243'] + data['hitop275']
    data['hitop_separation_insecurity_invcore'] = data['hitop50'] + data['hitop69'] + data['hitop81'] + data['hitop136'] + data['hitop151'] + data['hitop197']
    data['hitop_situational_phobia_invcore'] = data['hitop16'] + data['hitop165'] + data['hitop225'] + data['hitop247']
    data['hitop_social_anxiety_invcore'] = data['hitop1'] + data['hitop17'] + data['hitop114'] + data['hitop129'] + data['hitop204'] + data['hitop258']
    data['hitop_well_being_invcore'] = data['hitop9'] + data['hitop23'] + data['hitop149'] + data['hitop200'] + data['hitop244'] + data['hitop250'] + data['hitop281']
    data['hitop_all_depression_invcore'] = data.loc[:, dep_scales].sum(axis=1)
    data['hitop_all_anxiety_invcore'] = data.loc[:, anx_scales].sum(axis=1)


# Convergent validity hypotheses

In [5]:
# The p-value helps us determine whether or not we can meaningfully conclude that 
# the population correlation coefficient is different from zero, 
# based on what we observe from the sample.

"We will use these invariant scales in our subsequent tests of convergent and divergent validity. 

For scales in which we cannot establish an invariant core, we will test the relevant hypotheses in just the enriched sample."

In [6]:
def do_correlation(col1, col2):
    corr_result = pg.corr(col1, col2, method = 'spearman')
    #print(corr_result)
    my_r = corr_result.iloc[0]['r']
    my_r = "{:.2f}".format(float(my_r))
    my_ci = corr_result.iloc[0]['CI95']
    my_ci_1 = my_ci[0]
    my_ci_2 = my_ci[1]
    my_p = corr_result.iloc[0]['p_val']
    my_p = "{:.3f}".format(float(my_p)) 
    return(my_p, my_r, my_ci_1, my_ci_2)

In [7]:
def check_convergent_hypotheses(mydata=data_combined, mydata_gp=data_gp, mydata_en=data_en):
    '''
    tests convergent hypotheses; for hitop scales we use inv cores when appropriate
    '''
    
    print('\n --- CONV ANALYSIS START --- \n')
    
    print('\n-----> HYPOTHESIS C1 <-----\n')
    # HYPOTHESIS C1: hitop_anhedonic_depression will be positively correlated with PHQ-8
    # Spearman; alpha = .05
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_anhedonic_depression_invcore, mydata.phq_sum)
    if float(p) >= 0.05:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if float(r) > 0:
            print('Hyp C1 SUPPORTED: Anhedonic depression (or invariant core) positively correlates with PHQ-8')
        else:
            print('Hyp C1 NOT SUPPORTED: Anhedonic depression (or invariant core) DOES NOT positively correlate with PHQ-8')

    print('\n-----> HYPOTHESIS C2 <-----\n')      
    # HYPOTHESIS C2: hitop_well-being will be negatively correlated with PHQ-8
    # Spearman; alpha = .05
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_well_being_invcore, mydata.phq_sum)
    if float(p) >= 0.05:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if float(r) < 0:
            print('Hyp C2 SUPPORTED: Well-being (or invariant core) negatively correlates with PHQ-8')
        else:
            print('Hyp C2 NOT SUPPORTED: Well-being (or invariant core) DOES NOT negatively correlate with PHQ-8')

    print('\n-----> HYPOTHESIS C3 <-----\n')  
    # HYPOTHESIS C3: hitop_anxious_worry will be positively correlated with GAD-7
    # Spearman; alpha = .05
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_anxious_worry_invcore, mydata.gad_sum)
    if float(p) >= 0.05:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if float(r) > 0:
            print('Hyp C3 SUPPORTED: Anxious worry (or invariant core) positively correlates with GAD-7')
        else:
            print('Hyp C3 NOT SUPPORTED: Anxious worry (or invariant core) DOES NOT positively correlate with GAD-7') 

    print('\n-----> HYPOTHESIS C4 <-----\n')      
    # HYPOTHESIS C4: Higher sum of scores on the HiTOP Appetite Gain and Appetite Loss 
    # will be correlated with responses to PHQ-8 "Poor appetite or overeating"
    mydata['hitop_appetite_sum'] = mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss']
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_appetite_sum, mydata.phq_5)
    if float(p) >= 0.05:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        print('Hyp C4 NOT SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) DOES NOT correlate with PHQ item 5 - poor appetite or overeating')         
    else:
        if float(r) > 0:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            print('Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating')

    print('\n-----> HYPOTHESIS C5 <-----\n')      
    # HYPOTHESIS C5: the HiTOP Cognitive Problems will be positively correlated with responses 
    # to the PHQ-8 item "Trouble concentrating on things, such as school work, reading or watching television".
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_cognitive_problems, mydata.phq_7)
    if float(p) >= 0.05:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if float(r) > 0:
            print('Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with PHQ-8 item 7 "Trouble concentrating on things..."')
        else:
            print('Hyp C5 NOT SUPPORTED: HiTOP Cognitive problems DOES NOT positive correlate with PHQ-8 item 7 "Trouble concentrating on things..."')         

    print('\n-----> HYPOTHESIS C6 <-----\n')      
    # HYPOTHESIS C6: The HiTOP Insomnia subscale will be positively correlated with responses 
    # to the PHQ-8 item "Trouble falling or staying asleep, or sleeping too much"
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_insomnia, mydata.phq_3)        
    if float(p) >= 0.05:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if float(r) > 0:
            print('Hyp C6 SUPPORTED: HiTOP Insomnia positive correlates with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')
        else:
            print('Hyp C6 NOT SUPPORTED: HiTOP Insomnia DOES NOT positive correlate with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')         

    print('\n-----> HYPOTHESIS C7 <-----\n')      
    # HYPOTHESIS C7: Higher sum of all HiTOP subscales will be associated with greater likelihood
    # of being bothered by symptoms of a diagnosed mood or anxiety disorder. 

    # make sure hitop sum uses invcores (and doesnt include well-being)
    mydata['hitop_sum_new'] = mydata['hitop_anhedonic_depression_invcore'] + \
                                mydata['hitop_anxious_worry_invcore'] + \
                                mydata['hitop_appetite_gain_invcore'] + \
                                mydata['hitop_appetite_loss'] + \
                                mydata['hitop_cognitive_problems'] + \
                                mydata['hitop_hyposomnia'] + \
                                mydata['hitop_indecisiveness'] + \
                                mydata['hitop_insomnia'] + \
                                mydata['hitop_panic'] + \
                                mydata['hitop_separation_insecurity_invcore'] + \
                                mydata['hitop_shame_guilt'] + \
                                mydata['hitop_situational_phobia_invcore'] + \
                                mydata['hitop_social_anxiety_invcore']
    mydata['moodanxiety_bothered_int'] = mydata.moodanxiety_bothered.astype(int)
    # using an example from here: https://www.science.smith.edu/~jcrouser/SDS293/labs/lab4-py.html
    formula = 'moodanxiety_bothered_int ~ hitop_sum_new'        
    model = smf.glm(formula = formula, data=mydata, family=sm.families.Binomial())
    result = model.fit()
    print(result)
    print(result.summary())
    print(result.pvalues)
    p_val_hitop_sum = format(result.pvalues.iloc[1], 'f')
    if float(p_val_hitop_sum) >= 0.05:
        print(f"\nHitop sum of items is NOT a significant predictor of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
        print('Hyp C7 NOT SUPPORTED')         
    else:
        print(f"\nHitop sum of items IS A SIGNIFICANT PREDICTOR of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
        print('Hyp C7 SUPPORTED')
    print('some exploratory C7 tests')
    model = smf.glm(formula = formula, data=mydata.query('sample_type==1'), family=sm.families.Binomial())
    result = model.fit()
    print(result)
    print(result.summary())
    print(result.pvalues)
    model = smf.glm(formula = formula, data=mydata.query('sample_type==2'), family=sm.families.Binomial())
    result = model.fit()
    print(result)
    print(result.summary())
    print(result.pvalues)
    print('\n --- CONV ANALYSIS DONE ---')

In [8]:
check_convergent_hypotheses()


 --- CONV ANALYSIS START --- 


-----> HYPOTHESIS C1 <-----

Correlation Significant with p-val = 0.000 and r = 0.87(0.85, 0.89)
Hyp C1 SUPPORTED: Anhedonic depression (or invariant core) positively correlates with PHQ-8

-----> HYPOTHESIS C2 <-----

Correlation Significant with p-val = 0.000 and r = -0.53(-0.58, -0.48)
Hyp C2 SUPPORTED: Well-being (or invariant core) negatively correlates with PHQ-8

-----> HYPOTHESIS C3 <-----

Correlation Significant with p-val = 0.000 and r = 0.88(0.86, 0.89)
Hyp C3 SUPPORTED: Anxious worry (or invariant core) positively correlates with GAD-7

-----> HYPOTHESIS C4 <-----

Correlation Significant with p-val = 0.000 and r = 0.77(0.74, 0.8)
Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating

-----> HYPOTHESIS C5 <-----

Correlation Significant with p-val = 0.000 and r = 0.81(0.79, 0.83)
Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with P

In [9]:
check_convergent_hypotheses()


 --- CONV ANALYSIS START --- 


-----> HYPOTHESIS C1 <-----

Correlation Significant with p-val = 0.000 and r = 0.87(0.85, 0.89)
Hyp C1 SUPPORTED: Anhedonic depression (or invariant core) positively correlates with PHQ-8

-----> HYPOTHESIS C2 <-----

Correlation Significant with p-val = 0.000 and r = -0.53(-0.58, -0.48)
Hyp C2 SUPPORTED: Well-being (or invariant core) negatively correlates with PHQ-8

-----> HYPOTHESIS C3 <-----

Correlation Significant with p-val = 0.000 and r = 0.88(0.86, 0.89)
Hyp C3 SUPPORTED: Anxious worry (or invariant core) positively correlates with GAD-7

-----> HYPOTHESIS C4 <-----

Correlation Significant with p-val = 0.000 and r = 0.77(0.74, 0.8)
Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating

-----> HYPOTHESIS C5 <-----

Correlation Significant with p-val = 0.000 and r = 0.81(0.79, 0.83)
Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with P

In [10]:
with open(log_dir / "mylog_convergent.txt", "w") as f:
    with redirect_stdout(f):
        check_convergent_hypotheses()

# Divergent validity hypotheses

In [11]:
# This will need to SET THE SEED
random.seed(12345)
# test
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [12]:
dep_scales = [
    'hitop_anhedonic_depression_invcore',
    'hitop_appetite_gain_invcore',
    'hitop_appetite_loss',
    'hitop_cognitive_problems',
    'hitop_hyposomnia',
    'hitop_indecisiveness',
    'hitop_insomnia',
    'hitop_shame_guilt',
]
anx_scales = [
    'hitop_anxious_worry_invcore',
    'hitop_panic',
    'hitop_separation_insecurity_invcore',
    'hitop_situational_phobia_invcore',
    'hitop_social_anxiety_invcore'
]

In [13]:
mydata = data_combined.reset_index(drop=True)
mydata['gad_phq_sum'] = mydata['gad_sum'] + mydata['phq_sum']
mydata['hitop_sum_invcores'] = mydata['hitop_anhedonic_depression_invcore'] + mydata['hitop_anxious_worry_invcore'] + \
            mydata['hitop_appetite_gain_invcore'] + mydata['hitop_appetite_loss'] + mydata['hitop_cognitive_problems'] + \
            mydata['hitop_hyposomnia'] + mydata['hitop_indecisiveness'] + mydata['hitop_insomnia'] + mydata['hitop_panic'] + \
            mydata['hitop_separation_insecurity_invcore'] + mydata['hitop_shame_guilt'] + mydata['hitop_situational_phobia_invcore'] + \
            mydata['hitop_social_anxiety_invcore']
mydata['hitop_all_depression_invcore'] = mydata.loc[:, dep_scales].sum(axis=1)
mydata['hitop_all_axiety_invcore'] = mydata.loc[:, anx_scales].sum(axis=1)
mydata['baars_sum_d3']  = mydata.baars_sum.copy() # doubling baars_sum because it needs to be permuted for d1, but not for d


In [14]:
cols_to_z = [
    'gad_phq_sum',
    'hitop_sum_invcores',
    'baars_sum',
    'baars_sum_d3',
]
cols_for_d = [
    'gad_phq_sum',
    'hitop_sum_invcores',
    'baars_sum',
    'baars_sum_d3',
    'moodanxiety_bothered',
    'mood_bothered',
    'anxiety_bothered',
    'attention_bothered',
    'hitop_all_depression_invcore',
    'hitop_all_anxiety_invcore',
    'hitop_anhedonic_depression_invcore',
    'hitop_anxious_worry_invcore'
]
permed_cols = [
    'gad_phq_sum',
    'baars_sum',
    'moodanxiety_bothered',
    'attention_bothered',
    'mood_bothered',
    'anxiety_bothered'
]
nonpermed_cols = [cc for cc in cols_for_d if cc not in permed_cols]
nperms = 1000

In [15]:
ogddat = mydata.loc[:, ['caseid', 'sample_type'] + cols_for_d].copy()
ddat =  mydata.loc[:, ['caseid', 'sample_type'] + cols_for_d].copy()
ddat['pid'] = -1

In [16]:
def calc_dhs(df,pid=0, random_state=2342393490, cols_to_z=None):

    if cols_to_z is None:
        cols_to_z = [
            'gad_phq_sum',
            'hitop_sum_invcores',
            'baars_sum',
            'baars_sum_d3',
            'hitop_all_depression_invcore',
            'hitop_all_anxiety_invcore'
        ]
    df = df.copy()
    for cc in cols_to_z:
        df[cc] = (df[cc] - df[cc].mean()) / df[cc].std()
    # -----------------------------------------------------------------------
    # D1: There will be a stronger correlation between the sum of HiTOP scales and the sum of GAD-7 and PHQ-8 scores 
    #     than with BAARS-IV total score    
    gad_phq_v_hitop = stats.spearmanr(df.gad_phq_sum, df.hitop_sum_invcores)[0]
    baars_v_hitop = stats.spearmanr(df.baars_sum, df.hitop_sum_invcores)[0]
    d1 = gad_phq_v_hitop - baars_v_hitop
    
    # -----------------------------------------------------------------------
    # D2: The sum of HiTOP scales will be a better predictor of being bothered by symptoms of a diagnosed anxiety disorder 
    # than of being bothered by symptoms of ADHD.
    hitop_v_mab = mutual_info_classif(df.hitop_sum_invcores.values.reshape((-1,1)), df.moodanxiety_bothered.values, random_state=random_state)[0]
    hitop_v_ab = mutual_info_classif(df.hitop_sum_invcores.values.reshape((-1,1)), df.attention_bothered.values, random_state=random_state)[0]
    d2 = hitop_v_mab - hitop_v_ab
    # -----------------------------------------------------------------------
    # D3: The BAARS-IV total score will be a better predictor of being bothered by symptoms of ADHD 
    # than of a diagnosed mood or anxiety disorder.
    baars_v_ab = mutual_info_classif(df.baars_sum_d3.values.reshape((-1,1)), df.attention_bothered.values, random_state=random_state)[0]
    baars_v_mab = mutual_info_classif(df.baars_sum_d3.values.reshape((-1,1)), df.moodanxiety_bothered.values, random_state=random_state)[0]
    d3 = baars_v_ab - baars_v_mab
    # -----------------------------------------------------------------------
    # D4: The sum of HiTOP mood disorder scales will be a better predictor of being bothered by symptoms 
    # of a diagnosed mood disorder than of being bothered by symptoms of a diagnosed anxiety disorder.
    hitopdep_v_mb = mutual_info_classif(df.hitop_all_depression_invcore.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    hitopdep_v_ab = mutual_info_classif(df.hitop_all_depression_invcore.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    d4 =  hitopdep_v_mb - hitopdep_v_ab
    # exploratory 4, just use the anhedonic depression inv core
    hitopanhdep_v_mb = mutual_info_classif(df.hitop_anhedonic_depression_invcore.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    hitopanhdep_v_ab = mutual_info_classif(df.hitop_anhedonic_depression_invcore.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    exd4 =  hitopanhdep_v_mb - hitopanhdep_v_ab    
    # -----------------------------------------------------------------------
    # D5: The sum of HiTOP anxiety disorder scales will be a better predictor of being bothered by symptoms 
    # of a diagnosed anxiety disorder than of being bothered by symptoms of a diagnosed mood disorder.
    hitopanx_v_ab = mutual_info_classif(df.hitop_all_anxiety_invcore.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    hitopanx_v_mb = mutual_info_classif(df.hitop_all_anxiety_invcore.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    d5 = hitopanx_v_ab - hitopanx_v_mb
    # exploratory 5, just use the anxious worry inv core
    hitopanxwor_v_ab = mutual_info_classif(df.hitop_anxious_worry_invcore.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    hitopanxwor_v_mb = mutual_info_classif(df.hitop_anxious_worry_invcore.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    exd5 =  hitopanxwor_v_ab - hitopanxwor_v_mb    

    res = dict(
        pid=pid,
        gad_phq_v_hitop=gad_phq_v_hitop,
        baars_v_hitop=baars_v_hitop,
        d1=d1,
        hitop_v_mab=hitop_v_mab,
        hitop_v_ab=hitop_v_ab,
        d2=d2,
        baars_v_ab=baars_v_ab,
        baars_v_mab=baars_v_mab,
        d3=d3,
        hitopdep_v_mb=hitopdep_v_mb,
        hitopdep_v_ab=hitopdep_v_ab,
        d4=d4,
        hitopanx_v_ab=hitopanx_v_ab,
        hitopanx_v_mb=hitopanx_v_mb,
        d5=d5,
        hitopanhdep_v_mb=hitopanhdep_v_mb,
        hitopanhdep_v_ab=hitopanhdep_v_ab,
        exd4=exd4,
        hitopanxwor_v_ab=hitopanxwor_v_ab,
        hitopanxwor_v_mb=hitopanxwor_v_mb,
        exd5=exd5
    )
    return res

In [17]:
calc_dhs(ogddat, pid=-1)

{'pid': -1,
 'gad_phq_v_hitop': np.float64(0.9049998001185886),
 'baars_v_hitop': np.float64(0.8098369630375223),
 'd1': np.float64(0.09516283708106632),
 'hitop_v_mab': np.float64(0.20074181967689042),
 'hitop_v_ab': np.float64(0.060054394598890504),
 'd2': np.float64(0.14068742507799992),
 'baars_v_ab': np.float64(0.11670544710082709),
 'baars_v_mab': np.float64(0.18780185880044908),
 'd3': np.float64(-0.071096411699622),
 'hitopdep_v_mb': np.float64(0.18955764229871397),
 'hitopdep_v_ab': np.float64(0.16811309714401634),
 'd4': np.float64(0.021444545154697625),
 'hitopanx_v_ab': np.float64(0.20523747278330706),
 'hitopanx_v_mb': np.float64(0.2217671596490136),
 'd5': np.float64(-0.01652968686570655),
 'hitopanhdep_v_mb': np.float64(0.2043943452826935),
 'hitopanhdep_v_ab': np.float64(0.13712816382784032),
 'exd4': np.float64(0.06726618145485319),
 'hitopanxwor_v_ab': np.float64(0.1972003673648517),
 'hitopanxwor_v_mb': np.float64(0.2007971816330658),
 'exd5': np.float64(-0.003596814

In [18]:
perm_res = [calc_dhs(ogddat, pid=-1)]
for pid in range(nperms):
    permed_data = pd.concat((ddat.loc[:, nonpermed_cols], ddat.loc[:, ['sample_type'] + permed_cols].groupby("sample_type").sample(frac=1, replace=False).reset_index(drop=True)), axis=1)
    perm_res.append(calc_dhs(permed_data, pid=pid))
perm_res = pd.DataFrame(perm_res)

In [19]:
for dh in ['d1', 'd2', 'd3', 'd4', 'd5', 'exd4', 'exd5']:
    dhp = (perm_res.loc[0, dh] <= perm_res[dh]).mean()
    print(f"{dh}:{dhp}")

d1:0.000999000999000999
d2:0.000999000999000999
d3:0.978021978021978
d4:0.2547452547452547
d5:0.6453546453546454
exd4:0.000999000999000999
exd5:0.36363636363636365


# Do some bootstraps to get confidence intervals

In [20]:
nboots = 1000
bootdf = []
for bid in range(nboots):
    boot_data = ddat.groupby('sample_type').sample(frac=0.8, replace=False).reset_index()
    bootdf.append(calc_dhs(boot_data, pid=bid))
bootdf = pd.DataFrame(bootdf).rename(columns={'pid':'bid'})

In [21]:
bootdf.describe(percentiles=[0.025, 0.975])

,bid,gad_phq_v_hitop,baars_v_hitop,d1,hitop_v_mab,hitop_v_ab,d2,baars_v_ab,baars_v_mab,d3,hitopdep_v_mb,hitopdep_v_ab,d4,hitopanx_v_ab,hitopanx_v_mb,d5,hitopanhdep_v_mb,hitopanhdep_v_ab,exd4,hitopanxwor_v_ab,hitopanxwor_v_mb,exd5
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,499.500000,0.904872,0.809790,0.095083,0.201723,0.074402,0.127321,0.118045,0.197945,-0.079900,0.209916,0.174580,0.035336,0.197858,0.204563,-0.006705,0.194160,0.156821,0.037339,0.199120,0.185579,0.013541
std,288.819436,0.003629,0.006647,0.006005,0.014897,0.011585,0.018670,0.011856,0.017190,0.020250,0.016211,0.015894,0.018685,0.015290,0.015532,0.017187,0.017266,0.017226,0.019409,0.017306,0.017721,0.020021
min,0.000000,0.891354,0.789184,0.076356,0.151588,0.040086,0.071629,0.079040,0.143402,-0.142770,0.167540,0.123689,-0.036896,0.156626,0.159649,-0.053521,0.137854,0.109920,-0.025267,0.150110,0.130618,-0.061352
2.5%,24.975000,0.897873,0.796131,0.082682,0.171401,0.051221,0.089704,0.094674,0.166040,-0.118852,0.180363,0.143888,-0.001459,0.168343,0.176461,-0.040253,0.161896,0.126609,0.000163,0.167658,0.153353,-0.027393
97.5%,974.025000,0.912297,0.822516,0.106686,0.231038,0.096612,0.165169,0.142073,0.232827,-0.040338,0.241841,0.206018,0.073054,0.230620,0.237992,0.028852,0.228824,0.191509,0.074405,0.234165,0.222769,0.051117
max,999.000000,0.917169,0.836138,0.112685,0.250868,0.107276,0.192984,0.160494,0.261016,-0.019706,0.268567,0.221267,0.089020,0.253392,0.253520,0.045882,0.255176,0.211400,0.097100,0.266656,0.236593,0.085019


In [22]:
calc_dhs(ogddat, pid=-1)

{'pid': -1,
 'gad_phq_v_hitop': np.float64(0.9049998001185886),
 'baars_v_hitop': np.float64(0.8098369630375223),
 'd1': np.float64(0.09516283708106632),
 'hitop_v_mab': np.float64(0.20074181967689042),
 'hitop_v_ab': np.float64(0.060054394598890504),
 'd2': np.float64(0.14068742507799992),
 'baars_v_ab': np.float64(0.11670544710082709),
 'baars_v_mab': np.float64(0.18780185880044908),
 'd3': np.float64(-0.071096411699622),
 'hitopdep_v_mb': np.float64(0.18955764229871397),
 'hitopdep_v_ab': np.float64(0.16811309714401634),
 'd4': np.float64(0.021444545154697625),
 'hitopanx_v_ab': np.float64(0.20523747278330706),
 'hitopanx_v_mb': np.float64(0.2217671596490136),
 'd5': np.float64(-0.01652968686570655),
 'hitopanhdep_v_mb': np.float64(0.2043943452826935),
 'hitopanhdep_v_ab': np.float64(0.13712816382784032),
 'exd4': np.float64(0.06726618145485319),
 'hitopanxwor_v_ab': np.float64(0.1972003673648517),
 'hitopanxwor_v_mb': np.float64(0.2007971816330658),
 'exd5': np.float64(-0.003596814

In [23]:
bootdf.describe(percentiles=[0.025, 0.975]).iloc[[4,5], :]

,bid,gad_phq_v_hitop,baars_v_hitop,d1,hitop_v_mab,hitop_v_ab,d2,baars_v_ab,baars_v_mab,d3,hitopdep_v_mb,hitopdep_v_ab,d4,hitopanx_v_ab,hitopanx_v_mb,d5,hitopanhdep_v_mb,hitopanhdep_v_ab,exd4,hitopanxwor_v_ab,hitopanxwor_v_mb,exd5
2.5%,24.975,0.897873,0.796131,0.082682,0.171401,0.051221,0.089704,0.094674,0.166040,-0.118852,0.180363,0.143888,-0.001459,0.168343,0.176461,-0.040253,0.161896,0.126609,0.000163,0.167658,0.153353,-0.027393
97.5%,974.025,0.912297,0.822516,0.106686,0.231038,0.096612,0.165169,0.142073,0.232827,-0.040338,0.241841,0.206018,0.073054,0.230620,0.237992,0.028852,0.228824,0.191509,0.074405,0.234165,0.222769,0.051117


In [24]:
# This will need to SET THE SEED
random.seed(12345)
# test
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757
